# H3L1 — What two layers of attention do: induction heads

**Notebook 3 of this week.** We train small transformers on one task, look at
their attention weights, and see how two layers of attention solve it
together. The parts follow the slides:

1. the task: copying a sequence
2. a two-layer transformer, trained
3. accuracy against distance
4. layer 1: each token attends to the token before it
5. layer 2: the induction head
6. one layer is not enough

**Colab:** Runtime → Run all. About one minute on the CPU runtime; faster on
a GPU runtime.


In [ ]:
import math, os, time
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 13})          # readable on slides and in Colab
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0); np.random.seed(0)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)
C_BLUE, C_RED, C_TEAL, C_PURPLE, C_GRAY = \
    "#1F4E79", "#C00000", "#0E7C7B", "#7030A0", "#595959"
FIGDIR = os.environ.get("SEMINAR_FIGDIR")


def save(fig, name):
    """Only when the slides are built: store the figure for the deck."""
    if FIGDIR:
        for ext in ("pdf", "png"):
            fig.savefig(os.path.join(FIGDIR, f"{name}.{ext}"), bbox_inches="tight", dpi=200)

# ---------------------------------------------------------------- KNOB
SHOW_DISTANCE = 24     # the length of the example whose attention we plot
# ----------------------------------------------------------------------


## Part 1 — The task: copying a sequence

Each example is a random sequence of $m$ tokens, a separator, and the same
sequence again. The model is trained to predict the next token at every
position, but only the second copy is scored. To predict a token of the
second copy, the model must find where the current token appeared in the
first copy and read the token that followed it. The distance between a token
and its copy is $m + 1$, and $m$ changes from example to example (from 2 to
40), so no fixed offset works.

```
<bos>  t5 t61 t9 t44  <sep>  t5 t61 t9 t44
       |- first copy -|      |- predicted, token by token -|
```


In [ ]:
V, T, D_MODEL = 64, 96, 64          # vocabulary, sequence length, model width
BOS, SEP, PAD = 1, 2, 0
MAX_M = 40
STEPS, BATCH = 500, 64


def make_batch(n, m=None):
    """A random segment of length m, a separator, the segment again. The mask
    marks the positions whose next token is scored (the second copy)."""
    ms = torch.full((n,), m, dtype=torch.long) if m else torch.randint(2, MAX_M + 1, (n,))
    x = torch.full((n, T), PAD, dtype=torch.long)
    mask = torch.zeros(n, T - 1, dtype=torch.bool)
    x[:, 0] = BOS
    for i in range(n):
        mi = int(ms[i])
        seg = torch.randint(4, V, (mi,))
        x[i, 1:1 + mi] = seg
        x[i, 1 + mi] = SEP
        x[i, 2 + mi:2 + 2 * mi] = seg
        mask[i, 1 + mi:1 + 2 * mi] = True
    return x.to(DEV), mask.to(DEV)


x_demo, _ = make_batch(1, m=6)
print("one example:", x_demo[0, :16].tolist(), "...")
print("(BOS = 1, the sequence, SEP = 2, the sequence again, PAD = 0)")


## Part 2 — A two-layer transformer

The model has an embedding table, learned position vectors (taken as given
this week), two transformer blocks with four heads each, and an output layer
that gives a score for every token of the vocabulary. Each block is attention
followed by an MLP, both added to the residual stream, as in H2L1. We store
the attention weights of each block so that we can plot them.


In [ ]:
class Block(nn.Module):
    def __init__(self, H=4):
        super().__init__()
        self.H, self.dh = H, D_MODEL // H
        self.ln1, self.ln2 = nn.LayerNorm(D_MODEL), nn.LayerNorm(D_MODEL)
        self.wq, self.wk, self.wv, self.wo = (nn.Linear(D_MODEL, D_MODEL, bias=False)
                                              for _ in range(4))
        self.mlp = nn.Sequential(nn.Linear(D_MODEL, 4 * D_MODEL), nn.GELU(),
                                 nn.Linear(4 * D_MODEL, D_MODEL))
        self.attn = None                          # the last attention weights

    def forward(self, x):
        B, Tn, _ = x.shape
        h = self.ln1(x)
        split = lambda t: t.view(B, Tn, self.H, self.dh).transpose(1, 2)
        q, k, v = split(self.wq(h)), split(self.wk(h)), split(self.wv(h))
        s = q @ k.transpose(-1, -2) / math.sqrt(self.dh)
        s = s + torch.triu(torch.full((Tn, Tn), -torch.inf, device=x.device), 1)
        p = F.softmax(s, -1)
        self.attn = p.detach()
        x = x + self.wo((p @ v).transpose(1, 2).reshape(B, Tn, D_MODEL))
        return x + self.mlp(self.ln2(x))


class Transformer(nn.Module):
    def __init__(self, n_layers=2):
        super().__init__()
        self.emb = nn.Embedding(V, D_MODEL)
        self.pos = nn.Embedding(T, D_MODEL)       # learned positions
        self.blocks = nn.ModuleList(Block() for _ in range(n_layers))
        self.head = nn.Linear(D_MODEL, V)

    def forward(self, x):
        h = self.emb(x) + self.pos(torch.arange(x.shape[1], device=x.device))
        for b in self.blocks:
            h = b(h)
        return self.head(h)


def train(model, steps=STEPS, seed=1):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    losses = []
    for _ in range(steps):
        x, mask = make_batch(BATCH)
        logits = model(x)[:, :-1]
        target = x[:, 1:].clone()
        target[~mask] = -100                      # only the second copy is scored
        loss = F.cross_entropy(logits.reshape(-1, V), target.reshape(-1), ignore_index=-100)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    return losses


def accuracy(model, m, n=512):
    with torch.no_grad():
        x, mask = make_batch(n, m=m)
        pred = model(x)[:, :-1].argmax(-1)
        return (((pred == x[:, 1:]) & mask).sum() / mask.sum()).item()


two = Transformer(n_layers=2).to(DEV)
t0 = time.time()
losses = train(two)
n_par = sum(p.numel() for p in two.parameters())
print(f"parameters: {n_par:,}")
print(f"trained in {time.time() - t0:.0f} s")

fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.plot(np.convolve(losses, np.ones(20) / 20, "valid"), color=C_BLUE)
ax.axhline(math.log(V - 4), color=C_GRAY, ls=":", lw=1)
ax.text(0, math.log(V - 4) * 1.03, "chance", fontsize=8, color=C_GRAY)
ax.set_xlabel("training step"); ax.set_ylabel("loss on the second copy")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()


## Part 3 — Accuracy against distance

We measure the accuracy on new examples with the distance fixed. For a model
that finds the earlier position by its content, the distance should not
matter.


In [ ]:
dists = [2, 4, 8, 16, 24, 32, 40]
acc_two = [accuracy(two, m) for m in dists]
for m, a in zip(dists, acc_two):
    print(f"m = {m:2d}   accuracy {a:.2f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.2))
a1.plot(np.convolve(losses, np.ones(20) / 20, "valid"), color=C_BLUE)
a1.set_xlabel("training step"); a1.set_ylabel("loss on the second copy")
a2.bar([str(m) for m in dists], acc_two, color=C_BLUE)
a2.set_ylim(0, 1.05); a2.set_xlabel("length m"); a2.set_ylabel("accuracy")
for ax in (a1, a2):
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); save(fig, "fig-t-w5-h3l1-p3"); plt.show()


**Exercise.** Before running Part 3, predict whether the accuracy will fall as $m$ grows.

<details><summary>Answer</summary>

It does not fall much: 1.00 at $m = 2$ and 0.91 at $m = 40$. The model finds the earlier position by its content, wherever it is.

</details>


## Part 4 — Layer 1: each token attends to the token before it

We plot the attention weights on one example. In layer 1, one head puts
much of its weight just below the diagonal: each token attends mostly to the
token immediately before it. After this layer, every position also carries
information about its predecessor.


In [ ]:
x, _ = make_batch(1, m=SHOW_DISTANCE)
with torch.no_grad():
    two(x)
n = 2 + 2 * SHOW_DISTANCE
A1 = two.blocks[0].attn[0, :, :n, :n].cpu()           # [heads, n, n]
A2 = two.blocks[1].attn[0, :, :n, :n].cpu()

idx = torch.arange(1, n)
prev = A1[:, idx, idx - 1].mean(-1)     # per head
h1 = int(prev.argmax())
print("layer 1, previous token:")
print(" ".join(f"h{i} {v:.2f}" for i, v in enumerate(prev.tolist())))


## Part 5 — Layer 2: the induction head

In layer 2, one head looks from each position of the second copy back to the
position in the first copy that comes just **after** the earlier occurrence
of the current token. Its key there carries, from layer 1, the information
"my predecessor was the current token". The head copies the token found
there, which is the next token. On the plot this is a diagonal stripe at
distance $m$ from the main diagonal.

This two-step circuit, a previous-token head followed by a head that matches
on it, is called an **induction head**.


In [ ]:
m = SHOW_DISTANCE
q = torch.arange(1 + m, 1 + 2 * m)      # second copy
stripe = A2[:, q, q - m].mean(-1)       # per head
h2 = int(stripe.argmax())
print("layer 2, induction stripe:")
print(" ".join(f"h{i} {v:.2f}" for i, v in enumerate(stripe.tolist())))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.3))
for ax, M, title in [(axes[0], A1[h1], f"layer 1, head {h1}: the previous token"),
                     (axes[1], A2[h2], f"layer 2, head {h2}: the induction head")]:
    ax.imshow(M ** 0.5, cmap="Blues")                 # the square root shows faint weights
    ax.axvline(1 + SHOW_DISTANCE, color=C_GRAY, lw=0.6, ls=":")
    ax.axhline(1 + SHOW_DISTANCE, color=C_GRAY, lw=0.6, ls=":")
    ax.set_xlabel("key position"); ax.set_ylabel("query position")
    ax.set_title(title, fontsize=10)
fig.suptitle(f"attention weights on one example with m = {SHOW_DISTANCE} "
             "(dotted lines: the separator)", fontsize=10.5)
plt.tight_layout(); save(fig, "fig-t-w5-h3l1-p5"); plt.show()


**Exercise.** Set `SHOW_DISTANCE = 8` and run Part 5 again. Where does the stripe move?

<details><summary>Answer</summary>

The stripe moves to distance 8 from the diagonal, starting just after the separator. The head finds the earlier occurrence wherever it is.

</details>


## Part 6 — One layer is not enough

A single layer of attention can find the earlier occurrence of the current
token by matching contents, but the token it needs is the one **after** that
occurrence. To match on "my predecessor was the current token", the keys must
already contain the predecessor, which requires a previous layer. We train a
one-layer model with the same settings and compare.


In [ ]:
one = Transformer(n_layers=1).to(DEV)
train(one)
acc_one = [accuracy(one, m) for m in dists]
for m, a1, a2 in zip(dists, acc_one, acc_two):
    print(f"m = {m:2d}   one {a1:.2f}   two {a2:.2f}")

fig, ax = plt.subplots(figsize=(6.6, 3.6))
ax.plot(dists, acc_two, "o-", color=C_BLUE, label="two layers")
ax.plot(dists, acc_one, "s-", color=C_RED, label="one layer")
ax.axhline(1 / (V - 4), color=C_GRAY, ls=":", lw=1)
ax.text(dists[-1], 1 / (V - 4) + 0.03, "chance", fontsize=8, color=C_GRAY, ha="right")
ax.set_xlabel("length m of the copied sequence"); ax.set_ylabel("accuracy on the second copy")
ax.set_ylim(-0.04, 1.06)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); save(fig, "fig-t-w5-h3l1-p6"); plt.show()


**Exercise.** Train a model with three layers (`Transformer(n_layers=3)`). Is it better than the two-layer model, and would you expect it to be?

<details><summary>Answer</summary>

Slightly: 0.96 to 1.00 at every length, against 0.91 to 1.00 with two layers. The circuit needs two layers; a third adds capacity, not a new ability.

</details>


## What this notebook showed

| part | result |
|---|---|
| two layers | the copy task is solved at every distance |
| layer 1 | one head attends to the previous token |
| layer 2 | one head attends to the position after the earlier occurrence of the current token |
| one layer | much lower accuracy: the circuit needs two layers |

Induction heads appear in trained language models as well, and are one
mechanism behind in-context learning: continuing a pattern that appeared
earlier in the input.
